# QuKerNet 五十股截至 2026 年 3 月 25 日先导记录

本 Notebook 读取课程内已保存的汇总证据，不读取实验室行情，不连接无影，也不提交天衍任务。

学习目标：区分数据截止、特征筛选样本量、核训练样本量、评价覆盖以及真实执行预算。

In [ ]:
from pathlib import Path
import json, csv

anchor = Path.cwd().resolve()
candidates = [anchor, *anchor.parents]
root = next(p for p in candidates if (p / 'reports/qukernet_cloud_v3/cutoff_20260325/status.json').is_file())
report = root / 'reports/qukernet_cloud_v3/cutoff_20260325'
state = json.loads((report / 'status.json').read_text(encoding='utf-8'))
print(state['status_zh'])
print('实际行情截止：', state['timeline']['actual_cutoff'])

## 时间轴与信息可得性

重新按照验证开始之前的资料核实资格并抽取 50 只股票。将最新评价区间移动到历史截止之前后，旧版股票名单与特征选择不能直接复用。标签为次日开盘至第 6 个交易日开盘收益是否大于零；训练和验证边界按标签结束日净化。

In [ ]:
for key, value in state['timeline'].items():
    print(key, value)

## 数据门槛与观察记录

每个特征行要求此前 120 个市场日加当日连续有效行情。不通过固定样本门槛时不换股、不改变日期、不填补未知行情，也不启动云端计时。以下条数是保存记录；未知值不等于零。

本次三处历史缺口已与上市公司停复牌公告对应。正常停牌没有交易记录，不应要求供应商补造行情；这暴露了连续市场日规则与真实股票交易状态之间的适用性问题。公告定位见汇总中的 suspension_sources。

In [ ]:
for key, value in state['data'].items():
    print(key, value)
for source in state.get('suspension_sources', []):
    print(source['security'], source['announcement'], source['locator'], source['url'])
from IPython.display import Image, display
display(Image(filename=str(report / 'data_quality.png')))

## 固定实验与预算

8 比特、8 项特征、深度 1 至 3；16 条核训练和 16 条验证。QuKerNet 用 6 条线路生成标签，其中 4 条训练神经评分器、2 条诊断，再从 256 个经典预测候选中取 1 条实际评价。随机搜索独立评价 7 条。各进行 1 个配置、2 参数、2 步更新。

图为计划上限，不能读成实测成本。QuKerNet 额外的同配置初值评价单独计费。

In [ ]:
with (report / 'planned_budget.csv').open(encoding='utf-8-sig', newline='') as stream:
    for row in csv.DictReader(stream):
        print(row)
display(Image(filename=str(report / 'planned_budget.png')))
print('实际执行记录：', state['execution'])

## 结果应如何解释

KTA 提高只表明训练代理目标改变。预测效果须比较同配置初值、随机搜索、经典基线，并在仅 10 个评价日期的限制下解释。没有完整实测时，不绘制准确率提升曲线。

In [ ]:
if state['results']:
    for result in state['results']:
        print(result)
else:
    print('本版本没有已保存的模型效果结果，不能判断筛选或微调是否有效。')
assert state['planned_total_circuits'] == sum(x['planned_circuits'] for x in state['planned_budget'])
assert state['planned_total_requested_shots'] == 512 * state['planned_total_circuits']
if not state['data']['admitted']:
    assert state['execution']['new_cloud_circuits'] == 0
    assert state['execution']['new_window_started'] is False
print('汇总一致性检查通过。')

## 练习与验收

1. 说明为什么全历史用于特征筛选并不意味着量子核以大样本训练。
2. 按保存日期核对训练、验证、评价标签的结束时间。
3. 区分计划线路、真实请求 shots 与已核验结果；解释停止客户端不等于取消后台任务。
4. 在没有完整结果时写出可以支持和不能支持的结论。

验收：能从本 Notebook 的汇总证据复述实际状态；不将计划数或离线检查当作云端实测。

原创教学内容：Julius Zhou、AUFE、Cong Lei。教学文字与图表采用 CC BY 4.0，原创代码采用 MIT；第三方内容保留原许可。